In [5]:
from autogen import ConversableAgent, initiate_chats, AssistantAgent
import os
import pprint

In [2]:
# llm configuration for the agent
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": os.environ.get("GROQ_API_KEY"),
            "base_url": "https://api.groq.com/openai/v1",
            "api_type": "openai",
        }
    ],
    "cache_seed": None,
}

In [3]:
task = '''
        Write a concise but engaging blogpost about
       udemy.com. Make sure the blogpost is
       within 200 words.
       '''

In [6]:
writer = AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogposts (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [7]:
reply = writer.generate_reply(messages=[{"content": task, "role": "user"}])
print(reply)

[autogen.oai.client: 03-11 10:21:56] {744} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
"Unlock Your Potential with Udemy.com"

In today's fast-paced world, staying ahead of the curve is crucial. Udemy.com is an online learning platform that offers a vast array of courses to help you upskill and reskill. With over 130,000 courses to choose from, you can learn anything from programming and marketing to photography and cooking.

Udemy's courses are designed to be engaging and interactive, with video lectures, quizzes, and hands-on exercises. You can learn at your own pace, anytime and anywhere, making it perfect for busy professionals and students. The platform also offers courses in multiple languages, catering to a global audience.

Whether you're looking to boost your career or pursue a hobby, Udemy.com has something for everyone. With af

In [9]:
writer = AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogposts (with title) on given topics. You must polish your "
        "writing based on the feedback you receive from other agents and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

critic = AssistantAgent(
    name="Critic",
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [10]:
chat_result = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       udemy.com. Make sure the blogpost is
       within 200 words.
       

--------------------------------------------------------------------------------
[autogen.oai.client: 03-11 10:24:16] {744} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
Writer (to Critic):

"Unlock Your Potential with Udemy.com"

In today's fast-paced world, learning is no longer confined to traditional classrooms. Udemy.com has revolutionized the way we acquire new skills and knowledge. With over 130,000 courses to choose from, Udemy offers a vast array of subjects, from technology and business to creative arts and personal development.

Whether you're looking to enhance your career prospects, explore a new hobby, or simply broaden your horizons, Udemy has something for everyone. The platf

In [11]:
pprint.pprint(chat_result.summary)

('"Unlock Your Potential with Udemy.com"\n'
 '\n'
 'With over 130,000 courses to choose from, Udemy.com has become a leading '
 "platform for lifelong learning. I've personally used Udemy to enhance my "
 'skills in digital marketing, and the results have been impressive. Whether '
 "you're looking to boost your career or explore a new hobby, Udemy has "
 'something for everyone.\n'
 '\n'
 "The platform's user-friendly interface and excellent customer support make "
 'it easy to navigate and learn at your own pace. From technology and business '
 "to creative arts and personal development, Udemy's courses are taught by "
 'expert instructors and often priced affordably. With Udemy, you can take the '
 'first step towards achieving your goals and unlocking your full potential. '
 'Start learning today and discover a new skill or hobby - visit Udemy.com and '
 'browse their extensive course catalog to get started.')


In [14]:
# defining seo reviwer, legal reviwer and ethical reviwer
SEO_reviewer = AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

legal_reviewer = AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

ethics_reviewer = AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="""You are an ethics reviewer, known for 
        your ability to ensure that content is ethically sound 
        and free from any potential ethical issues. " 
        Make sure your suggestion is concise (within 3 bullet points), 
        concrete and to the point. "
        Begin the review by stating your role. """,
)

# meta reviwer

meta_reviewer = AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggregate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

### Chat orchestration

#### Nested chats

The way our chat is going to work is that when the writer will answer the critic, this time, the critic will actually trigger a series of nested chats with each specialized reviewer (Critic -> Reviewer and then Reviewer -> Critic). We are also going to request from each reviewer that they send back their review in a specific format. Each review will send back a LLM generated summary of their review in the following JSON format:  
`{'Reviewer': '', 'Review': ''}`  
This will make it easier for the meta-reviewer to summarize all reviews.

We will also define here is a simple function called `reflection_message()` that will create the following nessage:
```
Review the following content.

"BLOGPOST PROPOSED BY WRITER"
```
We will call this function to create the message sent by the Critic to each specialized reviewer sequentially.

In [12]:
# Nested chat orchetration
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

We are now going to define a new type of chat, a nested chat, that will trigger when the critic receives an answer. You can think about it like the inner monologue the Critic is having with other Reviewers that will help him provide the best possible criticism of the blogpost written by the reviewer. This is the structure our chat will follow:

**Main chat**:  
1. Critic -> Writer : Initial task (*"Write a concise but engaging blogpost ..."*)
2. Writer -> Critic : First version of the `blogpost`, this will trigger the **nested chat**

**Nested chat**:
1. Critic -> SEO reviewer: *"Review the following content: `blogpost`"*
2. SEO reviewer -> Critic: `SEO review` with context `{'Reviewer': '', 'Review': ''}` 
3. Critic -> Legal reviewer: *"Review the following content: `blogpost`"*
4. Legal reviewer -> Critic: `Legal review` with context `{'Reviewer': '', 'Review': ''}` 
5. Critic -> Ethics reviewer: *"Review the following content: `blogpost`"*
6. Ethics reviewer -> Critic: `Ethics review` with context `{'Reviewer': '', 'Review': ''}` 
7. Critic -> Meta reviewer: *"Aggregrate feedback from all reviewers and give final suggestions on the writing."*
8. Meta reviewer -> Critic: Summary of all reviews with all contexts `{'Reviewer': '', 'Review': ''}`

**Enf of nested chat**

**Back to the main chat**:
1. Critic -> Writer : Summary of all reviews with all contexts `{'Reviewer': '', 'Review': ''}`
2. Writer -> Critic : Refined version of the blogpost based on all reviews.

Since we've already seen how to define chats one by one, we'll define our nested chat all at once in a list this time:

In [15]:
review_chats = [ # This is our nested chat
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": 
        {
        "summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",
        },
     "max_turns": 1},
    
    {
     "recipient": legal_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    
    {"recipient": ethics_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
    
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
      "max_turns": 1},
]

* Note how the `message` for each nested chat is going to be constructed by the `reflection_message() function we previously defined
* Note how each specialized reviewer will send back their review in the requested JSON format `{'reviewer': '', 'review': ''}`

We now need to save and register this nested chat as a chat that will be **triggered when the writer will contact the critic**:

In [16]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

#### Main chat

Ok, we are now ready to start this chat. We will start this with the main chat that will trigger the Critic's nested chat as soon as the writed send back an first proposal blogpost answer to the critic.

Pay attention to the order in which the exchanges will happen and feel free to go back to the orchestration structure presented above to ensure that you understand how the nested chat works:

In [17]:
chat_results = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       udemy.com. Make sure the blogpost is
       within 200 words.
       

--------------------------------------------------------------------------------
[autogen.oai.client: 03-11 10:34:57] {744} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
Writer (to Critic):

"Unlock Your Potential with Udemy.com"

In today's fast-paced world, staying ahead of the curve is crucial. Udemy.com is a leading online learning platform that offers a vast array of courses to help you achieve your goals. With over 130,000 courses to choose from, you can acquire new skills, enhance your knowledge, and boost your career prospects.

From technology and business to creative skills and personal development, Udemy's courses are designed to cater to diverse interests and needs. The platform 